# Limpieza de archivos Álgebra

## 1. Cargar archivos de datos

In [1]:
from pathlib import Path
import pandas as pd
from utils_limpieza import *

# Solo archivos de calificaciones; el encabezado está en la fila 7 del Excel
archivos = sorted(Path("../data/algebra").glob("calificaciones*.xlsx"))
dfs = {a.name: pd.read_excel(a, header=6) for a in archivos}

## 2. Limpieza inicial de las calificaciones: nombres de columnas y filas nulas

In [2]:
for nombre, df in dfs.items():
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

    filas_nulas = df.isna().all(axis=1)
    sin_nombre = df[["apellido", "nombre"]].isna().any(axis=1) & ~filas_nulas
    sin_usuario = df["usuario"].isna() & ~filas_nulas
    print(nombre)
    print(f"  Filas totalmente nulas: {filas_nulas.sum()}")
    print(f"  Filas sin nombre/apellido: {sin_nombre.sum()}")
    print(f"  Filas sin usuario: {sin_usuario.sum()}")

    df = df.dropna(how="all").dropna(axis=1, how="all").dropna(subset=["usuario", "apellido", "nombre"])
    dfs[nombre] = filtrar_y_estandarizar_usuarios(df, nombre)

calificaciones_alumnos_20251024_114929.xlsx
  Filas totalmente nulas: 0
  Filas sin nombre/apellido: 0
  Filas sin usuario: 0
calificaciones_alumnos_20251024_114929.xlsx: staff/docentes excluidos = 0
calificaciones_alumnos_20251024_114941.xlsx
  Filas totalmente nulas: 0
  Filas sin nombre/apellido: 0
  Filas sin usuario: 0
calificaciones_alumnos_20251024_114941.xlsx: staff/docentes excluidos = 0
calificaciones_alumnos_20251024_114944.xlsx
  Filas totalmente nulas: 0
  Filas sin nombre/apellido: 0
  Filas sin usuario: 0
calificaciones_alumnos_20251024_114944.xlsx: staff/docentes excluidos = 0


## 3. Separación de la nota y clasificación del tipo de evaluación

In [3]:
tipos_evaluacion = {10: "autoevaluacion", 20: "recuperatorio", 25: "parcial"}

for nombre, df in dfs.items():
    # La nota máxima es la misma para todo el archivo: se toma de las notas no nulas
    nota_max = df["nota"].dropna().str.split("/").str[1].mode()[0]
    print(f"{nombre}: notas nulas = {df['nota'].isna().sum()}")
    df["nota"] = df["nota"].fillna(f"0/{nota_max}")

    df[["nota_obtenida", "nota_max"]] = df["nota"].str.split("/", expand=True).astype(float)
    df["tipo_evaluacion"] = df["nota_max"].map(tipos_evaluacion)
    dfs[nombre] = validar_notas(df, nombre, set(tipos_evaluacion))

calificaciones_alumnos_20251024_114929.xlsx: notas nulas = 0
calificaciones_alumnos_20251024_114929.xlsx: nota máxima inesperada = 0, notas fuera de rango = 0
calificaciones_alumnos_20251024_114941.xlsx: notas nulas = 0
calificaciones_alumnos_20251024_114941.xlsx: nota máxima inesperada = 0, notas fuera de rango = 0
calificaciones_alumnos_20251024_114944.xlsx: notas nulas = 0
calificaciones_alumnos_20251024_114944.xlsx: nota máxima inesperada = 0, notas fuera de rango = 0


## 4. Eliminación de duplicados, unificación de calificaciones y exportación a Parquet

In [4]:
for nombre, df in dfs.items():
    duplicadas = df.duplicated().sum()
    usuarios_repetidos = df["usuario"].duplicated().sum()
    print(f"{nombre}: filas duplicadas = {duplicadas}, usuarios repetidos = {usuarios_repetidos}")

    # Se eliminan filas idénticas y, si un usuario aparece más de una vez en la misma evaluación,
    # se conserva su mejor nota
    dfs[nombre] = (df.drop_duplicates()
                   .sort_values("nota_obtenida", ascending=False)
                   .drop_duplicates(subset="usuario")
                   .sort_index())

# Se guarda el archivo de origen para distinguir evaluaciones del mismo tipo (ej. autoevaluación 1 y 2)
df_calificaciones = pd.concat(dfs, names=["archivo"]).reset_index(level="archivo").reset_index(drop=True)
df_calificaciones = df_calificaciones.drop(columns=["nota"], errors="ignore")

df_calificaciones.to_parquet("../data/algebra/calificaciones_algebra.parquet", index=False)
df_calificaciones

calificaciones_alumnos_20251024_114929.xlsx: filas duplicadas = 0, usuarios repetidos = 0
calificaciones_alumnos_20251024_114941.xlsx: filas duplicadas = 0, usuarios repetidos = 0
calificaciones_alumnos_20251024_114944.xlsx: filas duplicadas = 0, usuarios repetidos = 0


,archivo,usuario,apellido,nombre,observaciones,nota_obtenida,nota_max,tipo_evaluacion
0,calificaciones_alumnos_20251024_114929.xlsx,ALU_1001,Gómez,Manuel,aprobada,9.0,10.0,autoevaluacion
1,calificaciones_alumnos_20251024_114929.xlsx,ALU_1004,Flores,Gabriel,aprobada,9.0,10.0,autoevaluacion
2,calificaciones_alumnos_20251024_114929.xlsx,ALU_1005,Vega,Agustín,aprobada,9.0,10.0,autoevaluacion
3,calificaciones_alumnos_20251024_114929.xlsx,ALU_1006,Ramos,Patricio,aprobada,10.0,10.0,autoevaluacion
4,calificaciones_alumnos_20251024_114929.xlsx,ALU_1008,Ramírez,Gabriel,aprobada,10.0,10.0,autoevaluacion
...,...,...,...,...,...,...,...,...
474,calificaciones_alumnos_20251024_114944.xlsx,ALU_1312,Rodríguez,Juan,aprobada,16.0,25.0,parcial
475,calificaciones_alumnos_20251024_114944.xlsx,ALU_1314,Pérez,Lucía,aprobada,25.0,25.0,parcial
476,calificaciones_alumnos_20251024_114944.xlsx,ALU_1315,Flores,Felipe,aprobada,21.0,25.0,parcial
477,calificaciones_alumnos_20251024_114944.xlsx,ALU_1316,Sosa,Emilia,aprobada,17.0,25.0,parcial


## 5. Carga y limpieza del reporte de participación

In [5]:
archivo = next(Path("../data/algebra").glob("reporte_participacion*.xlsx"))
df_participacion = pd.read_excel(archivo, header=[5, 6])
# Encabezado de dos niveles (ej. "Encuestas" / "Cantidad"): se unen en un solo nombre
df_participacion.columns = pd.Index(
    [" ".join(n for n in col if not n.startswith("Unnamed")) for col in df_participacion.columns]
)

df_participacion = limpiar_reporte(df_participacion, "participacion")
df_participacion = procesar_participacion(df_participacion)

df_participacion.to_parquet("../data/algebra/participacion_algebra.parquet", index=False)
df_participacion

participacion: staff/docentes excluidos = 10
participacion:
  Fechas inconsistentes (última < primera, o primera < alta): 0
  Filas con valores numéricos negativos: 0
  Avance fuera de 0-100: 0
  Aulas: ['TUCD-MD-Álgebra 2025']
  Perfiles: {'Alumnos': 319}
  Filas duplicadas: 0, usuarios repetidos: 0
  Nulos por columna: {'primera_interaccion': 51, 'ultima_interaccion': 51}


,usuario,apellido,nombre,aula,perfil,fecha_alta,avance,interacciones,primera_interaccion,ultima_interaccion,...,foros_temas_de_debate_con_intervencion,foros_cantidad_de_intervenciones_del_usuario,anuncios_cantidad,anuncios_cargados,noticias_cantidad,noticias_cargados,tiempo_estimado_de_uso_min,fechas_inconsistentes,valores_negativos,avance_fuera_de_rango
0,ALU_1001,Gómez,Manuel,TUCD-MD-Álgebra 2025,Alumnos,2025-08-05,86,1015,2025-08-05 10:50:31,2025-10-23 21:47:55,...,0,0,0,0,9,0,607.0,False,False,False
1,ALU_1002,Silva,Diego,TUCD-MD-Álgebra 2025,Alumnos,2025-08-05,90,616,2025-08-05 14:16:21,2025-10-23 19:44:57,...,0,0,0,0,9,0,389.0,False,False,False
2,ALU_1003,Castro,Federico,TUCD-MD-Álgebra 2025,Alumnos,2025-08-05,0,0,NaT,NaT,...,0,0,0,0,9,0,0.0,False,False,False
3,ALU_1004,Flores,Gabriel,TUCD-MD-Álgebra 2025,Alumnos,2025-08-05,95,633,2025-08-05 11:01:59,2025-10-22 00:08:08,...,0,0,0,0,9,0,514.0,False,False,False
4,ALU_1005,Vega,Agustín,TUCD-MD-Álgebra 2025,Alumnos,2025-08-05,80,410,2025-08-05 11:47:53,2025-10-24 10:10:42,...,0,0,0,0,9,0,300.0,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
314,ALU_1315,Flores,Felipe,TUCD-MD-Álgebra 2025,Alumnos,2025-08-05,90,320,2025-08-05 22:52:07,2025-10-21 20:16:58,...,0,0,0,0,9,0,147.0,False,False,False
315,ALU_1316,Sosa,Emilia,TUCD-MD-Álgebra 2025,Alumnos,2025-08-05,80,204,2025-08-11 19:43:35,2025-10-19 22:33:24,...,0,0,0,0,9,0,71.0,False,False,False
316,ALU_1317,Morales,Nahuel,TUCD-MD-Álgebra 2025,Alumnos,2025-08-05,27,38,2025-08-11 18:00:00,2025-09-18 10:40:29,...,0,0,0,0,9,0,9.0,False,False,False
317,ALU_1318,Villalba,Candelaria,TUCD-MD-Álgebra 2025,Alumnos,2025-08-05,40,63,2025-08-11 13:55:27,2025-09-15 17:48:49,...,0,0,0,0,9,0,43.0,False,False,False


## 6. Carga y limpieza del reporte de trazabilidad

In [6]:
archivo = next(Path("../data/algebra").glob("reporte_trazabilidad*.xlsx"))
df_trazabilidad = procesar_trazabilidad(limpiar_reporte(pd.read_excel(archivo, header=5), "trazabilidad"))

df_trazabilidad.to_parquet("../data/algebra/trazabilidad_algebra.parquet", index=False)
df_trazabilidad

trazabilidad: staff/docentes excluidos = 3734
trazabilidad:
  Eventos sin fecha válida (descartados): 0
  Rango de fechas: 2025-08-05 10:19:14 -> 2025-10-24 10:16:39
  Aulas: ['TUCD-MD-Álgebra 2025']
  Perfiles: {'Alumnos': 41983}
  Filas duplicadas (descartadas): 190
  Nulos por columna: {'accion': 4}


,fecha,ip_usuario,usuario,apellido,nombre,aula,perfil,seccion,accion,descripcion
0,2025-08-05 10:50:31,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Álgebra 2025,Alumnos,Inicio,Acceder,Acceso a la sección 'Inicio'
1,2025-08-11 13:29:48,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Álgebra 2025,Alumnos,Inicio,Acceder,Acceso a la sección 'Inicio'
2,2025-08-11 13:29:57,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Álgebra 2025,Alumnos,Presentación,Acceder,Acceso a la sección 'Presentación'
3,2025-08-11 13:30:01,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Álgebra 2025,Alumnos,Presentación,Ver tópico,Visualización del topico '01- Presentación'
4,2025-08-11 13:30:35,10.10.3.252,ALU_1001,Gómez,Manuel,TUCD-MD-Álgebra 2025,Alumnos,Programa,Acceder,Acceso a la sección 'Programa'
...,...,...,...,...,...,...,...,...,...,...
41788,2025-10-13 20:09:09,10.10.4.25,ALU_1319,Maldonado,Carlos,TUCD-MD-Álgebra 2025,Alumnos,Programa > Material de estudio,Descargar,Descarga del material de estudio 'TP_4.pdf' de...
41789,2025-10-13 20:12:31,10.10.4.25,ALU_1319,Maldonado,Carlos,TUCD-MD-Álgebra 2025,Alumnos,Programa,Acceder,Acceso a la sección 'Programa'
41790,2025-10-13 20:12:40,10.10.4.25,ALU_1319,Maldonado,Carlos,TUCD-MD-Álgebra 2025,Alumnos,Programa > Textos,Ver texto,Acceso al texto 'Link de encuentros Sincrónico...
41791,2025-10-19 11:53:40,10.10.4.25,ALU_1319,Maldonado,Carlos,TUCD-MD-Álgebra 2025,Alumnos,Inicio,Acceder,Acceso a la sección 'Inicio'


## 7. Cruce con Matriculados

In [7]:
df_matriculados = pd.read_parquet("../data/matriculados/matriculados.parquet")
df_cruce = cruzar_con_matriculados(df_matriculados, "algebra", {
    "trazabilidad": df_trazabilidad,
    "participacion": df_participacion,
    "calificaciones": df_calificaciones,
})
df_cruce.to_parquet("../data/algebra/cruce_matriculados_algebra.parquet", index=False)
df_cruce[df_cruce["activo_no_inscripto"]]

Matriculados (algebra): 165
  trazabilidad: 268 usuarios, 105 no matriculados
  participacion: 319 usuarios, 154 no matriculados
  calificaciones: 196 usuarios, 52 no matriculados
Activos no inscriptos: 105
Activos con estado Baja: 2
Inscriptos sin actividad: 2


,usuario,matriculado,en_trazabilidad,en_participacion,en_calificaciones,estado_insc,instancia,activo_no_inscripto,activo_con_baja,inscripto_sin_actividad
4,ALU_1005,False,True,True,True,<NA>,<NA>,True,False,False
8,ALU_1009,False,True,True,True,<NA>,<NA>,True,False,False
9,ALU_1010,False,True,True,False,<NA>,<NA>,True,False,False
10,ALU_1011,False,True,True,False,<NA>,<NA>,True,False,False
11,ALU_1012,False,True,True,True,<NA>,<NA>,True,False,False
...,...,...,...,...,...,...,...,...,...,...
310,ALU_1311,False,True,True,False,<NA>,<NA>,True,False,False
314,ALU_1315,False,True,True,True,<NA>,<NA>,True,False,False
316,ALU_1317,False,True,True,False,<NA>,<NA>,True,False,False
317,ALU_1318,False,True,True,False,<NA>,<NA>,True,False,False


## 8. Verificar nulos y observaciones

In [8]:
sin_fecha = df_participacion["primera_interaccion"].isna()
print("Sin fecha pero con interacciones:", int((sin_fecha & (df_participacion["interacciones"] > 0)).sum()))
print("Con fecha pero 0 interacciones:", int((~sin_fecha & (df_participacion["interacciones"] == 0)).sum()))

con_actividad = set(df_participacion.loc[df_participacion["interacciones"] > 0, "usuario"])
print("Con interacciones pero sin trazabilidad:", len(con_actividad - set(df_trazabilidad["usuario"])))
print("En trazabilidad pero con 0 interacciones:", len(set(df_trazabilidad["usuario"]) - con_actividad))

print(df_calificaciones["observaciones"].value_counts(dropna=False))

Sin fecha pero con interacciones: 0
Con fecha pero 0 interacciones: 0
Con interacciones pero sin trazabilidad: 0
En trazabilidad pero con 0 interacciones: 0
observaciones
aprobada       478
no aprobada      1
Name: count, dtype: int64
